In [ ]:
# selected kernel: /project/spott/schan/envs/dimelo/bin/python

from dimelo import parse_bam


In [1]:
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import urllib, gzip
import sys
import os
from pathlib import Path
import datetime
import platform
import shutil

# Initialization
set path to fiber-seq bam files


In [ ]:
# base input and output paths
input_dir = Path("/project/spott/1_Shared_projects/LCL_Fiber_seq/preprocess_final_merged_samples/")
output_dir = Path("/project/spott/cshan/fiber-seq/results/modkit_converted/")

output_dir.mkdir(exist_ok=True)

# input bam file
## test one sample first
sample = "AL10_bc2178_19130.5mC.6mA.aligned.phased.bam"
bam_file = os.path.join(input_dir, sample)

# remove ".bam" suffix and add ".mk.bam"
converted_bam = os.path.join(
    output_dir,
    sample.replace(".bam", ".mk.bam"))


# loop over /project/spott/1_Shared_projects/LCL_Fiber_seq/preprocess_final_merged_samples/*.bam

print(converted_bam)


# Modkit
https://github.com/streetslab/dimelo-toolkit/tree/main?tab=readme-ov-file#Basic-Use 

modkit: modify tags in bam file to be compatible with dimelo parsing


for pacbio, mode = implicit
for nanopore, mode = ambiguous


In [ ]:
#!/bin/bash
#SBATCH --job-name=modkit_convert
#SBATCH --time=12:00:00
#SBATCH --mem=100G
#SBATCH --cpus-per-task=6
#SBATCH --account=pi-spott
#SBATCH --partition=caslake
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/modkit_convert_%A_%a.out

module load samtools
module load python/miniforge-25.3.0
source activate /project/spott/cshan/envs/dimelo

bam_file="/project/spott/1_Shared_projects/LCL_Fiber_seq/preprocess_final_merged_samples/AL10_bc2178_19130.5mC.6mA.aligned.phased.bam"
converted_bam="/project/spott/cshan/fiber-seq/results/modkit_converted/AL10_bc2178_19130.5mC.6mA.aligned.phased.mk.bam"

modkit update-tags \
    --mode implicit \
    --threads 8 \
    "$bam_file" \
    "$converted_bam"

samtools index "$converted_bam"


In [ ]:
# run script: /project/spott/cshan/fiber-seq/code/modkit_update_tags.sh
!sbatch /project/spott/cshan/fiber-seq/code/modkit_update_tags.sh


In [ ]:
# index the converted bam file with samtools
import pysam
pysam.index(converted_bam)


# parsing the updated bam file for both pile up and read extraction


In [ ]:
# define parameters for parsing
ref_genome_file='/project/spott/reference/human/GRCh38/genome/hg38.fa'
bam_file='/project/spott/cshan/fiber-seq/results/modkit_converted/AL10_bc2178_19130.5mC.6mA.aligned.phased.mk.bam'
output_dir="/project/spott/cshan/fiber-seq/results/modkit_converted"

 # parsing can optionally specify mod codes. 
    # The default is Y or a for adenine and m or Z for cytosine, corresponding to methylation
motifs = ['A,0','CG,0']

parse_prefix = 'parsed'
met_thresh = 190
ncore = 4

print("Parsing bam file with dimelo...")

pileup_file, pileup_regions = parse_bam.pileup(
    input_file = bam_file,
    output_name = parse_prefix,
    ref_genome = ref_genome_file,
    output_directory = output_dir,
    ## optional bed file of regions to parse. If not provided, will parse whole genome
    # regions = regions_bed, 
    motifs = motifs,
    thresh = met_thresh,
    # window_size = window_size,
    cores = ncore, 
    quiet = True,
    cleanup = True,
    override_checks = True, # override bam format and reference alignment checking
    log=True,
)


In [ ]:
# extraction
extract_file, extract_regions = parse_bam.extract(
    input_file=bam_file,
    output_name=parse_prefix,
    ref_genome=ref_genome_file,
    output_directory=output_dir,
    # regions=regions_bed,
    motifs= motifs,
    thresh = met_thresh,
    # window_size = window_size,
    cores = ncore, 
    quiet = True,
    cleanup = False,
    override_checks = True, # override bam format and reference alignment checking
    log=True,
)


wrapper to parse and extract bam files, submit job as slurm script

pile up

In [ ]:
#!/bin/bash
#SBATCH --job-name=dimelo_parse_pileup
#SBATCH --time=12:00:00
#SBATCH --mem=120G
#SBATCH --cpus-per-task=6
#SBATCH --account=pi-spott
#SBATCH --partition=caslake
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/dimelo_parse_pileup_%j.out

module load python/miniforge-25.3.0
source activate /project/spott/cshan/envs/dimelo

python - <<'PY'
import os
from dimelo import parse_bam

# define parameters for parsing
ref_genome_file = '/project/spott/reference/human/GRCh38/genome/hg38.fa'
bam_file = '/project/spott/cshan/fiber-seq/results/modkit_converted/AL10_bc2178_19130.5mC.6mA.aligned.phased.mk.bam'
output_dir = '/project/spott/cshan/fiber-seq/results/modkit_converted'

# parsing can optionally specify mod codes.
# default is Y/a for adenine and m/Z for cytosine methylation
motifs = ['A,0', 'CG,0']

parse_prefix = 'parsed'
met_thresh = 190
ncore=1

print(f"Running dimelo with cores={ncore}")
print("Running dimelo pileup...")

pileup_file, pileup_regions = parse_bam.pileup(
    input_file=bam_file,
    output_name=parse_prefix,
    ref_genome=ref_genome_file,
    output_directory=output_dir,
    # regions=regions_bed,
    motifs=motifs,
    thresh=met_thresh,
    # window_size=window_size,
    cores=ncore,
    quiet=True,
    cleanup=True,
    override_checks=True,
    log=True,
)

print("pileup_file:", pileup_file)
print("Expected output directory:", f"{output_dir}/{parse_prefix}")
PY

In [ ]:
!sbatch /project/spott/cshan/fiber-seq/code/dimelo_parse_pileup.sh

extract

In [ ]:
#!/bin/bash
#SBATCH --job-name=dimelo_parse_extract
#SBATCH --time=12:00:00
#SBATCH --mem=180G
#SBATCH --cpus-per-task=6
#SBATCH --account=pi-spott
#SBATCH --partition=caslake
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/dimelo_parse_extract_%j.out



module load python/miniforge-25.3.0
source activate /project/spott/cshan/envs/dimelo

python - <<'PY'
import os
import shutil
from dimelo import parse_bam

# define parameters for parsing
ref_genome_file = '/project/spott/reference/human/GRCh38/genome/hg38.fa'
bam_file = '/project/spott/cshan/fiber-seq/results/modkit_converted/AL10_bc2178_19130.5mC.6mA.aligned.phased.mk.bam'
output_dir = '/project/spott/cshan/fiber-seq/results/modkit_converted'

# parsing can optionally specify mod codes.
# default is Y/a for adenine and m/Z for cytosine methylation
motifs = ['A,0', 'CG,0']

parse_prefix = 'parsed'
met_thresh = 190
ncore=1

print("Running dimelo extract...")

extract_file, extract_regions = parse_bam.extract(
    input_file=bam_file,
    output_name=parse_prefix,
    ref_genome=ref_genome_file,
    output_directory=output_dir,
    # regions=regions_bed,
    motifs=motifs,
    thresh=met_thresh,
    # window_size=window_size,
    cores=ncore,
    quiet=True,
    cleanup=True,
    override_checks=True,
    log=True,
)

print("extract_file:", extract_file)
print("Expected output directory:", f"{output_dir}/{parse_prefix}")
PY

In [3]:
!sbatch /project/spott/cshan/fiber-seq/code/dimelo_parse_extract.sh



sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: caslake
sbatch: QOS-Flag: caslake
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***
Submitted batch job 48080196
